In [0]:
dbutils.secrets.listScopes()

In [0]:
display(
    dbutils.fs.ls("abfss://raw@strohitattrition01.dfs.core.windows.net/")
)

In [0]:
from pyspark.sql.functions import input_file_name

day2_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("abfss://raw@strohitattrition01.dfs.core.windows.net/day2/") \
    .withColumn("source_file", input_file_name())

display(day2_df)

In [0]:
silver_df = spark.table("silver_employee")

display(silver_df)

In [0]:
print("Day 2 Records :", day2_df.count())
print("Silver Records :", silver_df.count())

In [0]:
display(
    silver_df.groupBy("source_file").count()
)

In [0]:
from pyspark.sql.functions import regexp_extract

display(
    silver_df.withColumn(
        "day",
        regexp_extract("source_file", r"(day\d+)", 1)
    ).groupBy("day").count()
)

In [0]:
from pyspark.sql.functions import col, lit, regexp_extract

In [0]:
silver_df = spark.table("silver_employee")

print("Silver Records:", silver_df.count())

In [0]:
from pyspark.sql.functions import regexp_extract, lit

day1_df = (
    silver_df
    .filter(
        regexp_extract("source_file", r"(day\d+)", 1) == "day1"
    )
    .withColumn("snapshot_day", lit(1))
)

display(day1_df)

In [0]:
day2_df = (
    silver_df
    .filter(
        regexp_extract("source_file", r"(day\d+)", 1) == "day2"
    )
    .withColumn("snapshot_day", lit(2))
)

print("Day 2 Records:", day2_df.count())
display(day2_df)

In [0]:
silver_df = spark.table("silver_employee")

print("Silver Records:", silver_df.count())

In [0]:
scd2_day1 = (
    day1_df
    .withColumn("effective_from_day", col("snapshot_day"))
    .withColumn("effective_to_day", lit(9999))
    .withColumn("is_current", lit(True))
)

display(scd2_day1)

In [0]:
spark.sql("DROP TABLE IF EXISTS silver_employee_scd2")

In [0]:
scd2_day1.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_employee_scd2")

In [0]:
scd2_df = spark.table("silver_employee_scd2")

print("SCD2 Records:", scd2_df.count())

display(scd2_df)

In [0]:
current_df = (
    scd2_df
    .filter(col("is_current") == True)
)

print("Current Records:", current_df.count())

In [0]:
comparison_df = (
    day2_df.alias("new")
    .join(
        current_df.alias("old"),
        col("new.emp_id") == col("old.emp_id"),
        "left"
    )
)

In [0]:
comparison_df = comparison_df.withColumn(
    "record_changed",
    col("old.emp_id").isNull() |
    (col("new.name") != col("old.name")) |
    (col("new.department") != col("old.department")) |
    (col("new.salary") != col("old.salary")) |
    (col("new.join_date") != col("old.join_date")) |
    (col("new.status") != col("old.status"))
)

display(
    comparison_df
    .filter(col("record_changed") == True)
)

In [0]:
print(
    "New / Changed Employees:",
    comparison_df
    .filter(col("record_changed") == True)
    .count()
)

In [0]:
changed_df = comparison_df.filter(
    col("record_changed") == True
)

display(changed_df)

In [0]:
from delta.tables import DeltaTable

# Select only the Day 2 columns we need
changed_df_clean = comparison_df.filter(
    col("record_changed") == True
).select(
    col("new.emp_id").alias("emp_id"),
    col("new.name").alias("name"),
    col("new.department").alias("department"),
    col("new.salary").alias("salary"),
    col("new.join_date").alias("join_date"),
    col("new.status").alias("status"),
    col("new.source_file").alias("source_file"),
    col("new.snapshot_day").alias("snapshot_day")
)

print("Records to update:", changed_df_clean.count())

In [0]:
delta_table = DeltaTable.forName(
    spark,
    "silver_employee_scd2"
)

delta_table.alias("target").merge(
    changed_df_clean.alias("source"),
    "target.emp_id = source.emp_id AND target.is_current = true"
).whenMatchedUpdate(
    set={
        "effective_to_day": "source.snapshot_day - 1",
        "is_current": "false"
    }
).execute()

In [0]:
new_versions = (
    changed_df_clean
    .withColumn("effective_from_day", col("snapshot_day"))
    .withColumn("effective_to_day", lit(9999))
    .withColumn("is_current", lit(True))
    .drop("snapshot_day")
)

new_versions.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("silver_employee_scd2")

In [0]:
scd2_df = spark.table("silver_employee_scd2")

print("Total SCD2 Records:", scd2_df.count())

display(
    scd2_df
    .orderBy("emp_id", "effective_from_day")
)

In [0]:
display(
    scd2_df
    .filter(col("emp_id") == 1)
    .orderBy("effective_from_day")
)

In [0]:
print("Historical records:",
      scd2_df.filter(col("is_current") == False).count())

print("Current records:",
      scd2_df.filter(col("is_current") == True).count())

In [0]:
day3_df = (
    silver_df
    .filter(
        regexp_extract("source_file", r"(day\d+)", 1) == "day3"
    )
    .withColumn("snapshot_day", lit(3))
)

print("Day 3 Records:", day3_df.count())

In [0]:
scd2_df = spark.table("silver_employee_scd2")

current_df = scd2_df.filter(
    col("is_current") == True
)

print("Current Records:", current_df.count())

In [0]:
comparison_df = (
    day3_df.alias("new")
    .join(
        current_df.alias("old"),
        col("new.emp_id") == col("old.emp_id"),
        "left"
    )
)

In [0]:
comparison_df = comparison_df.withColumn(
    "record_changed",
    col("old.emp_id").isNull() |
    (col("new.name") != col("old.name")) |
    (col("new.department") != col("old.department")) |
    (col("new.salary") != col("old.salary")) |
    (col("new.join_date") != col("old.join_date")) |
    (col("new.status") != col("old.status"))
)

print(
    "New / Changed Employees:",
    comparison_df
    .filter(col("record_changed") == True)
    .count()
)

In [0]:
changed_df_clean = comparison_df.filter(
    col("record_changed") == True
).select(
    col("new.emp_id").alias("emp_id"),
    col("new.name").alias("name"),
    col("new.department").alias("department"),
    col("new.salary").alias("salary"),
    col("new.join_date").alias("join_date"),
    col("new.status").alias("status"),
    col("new.source_file").alias("source_file"),
    col("new.snapshot_day").alias("snapshot_day")
)

print("Records to update:", changed_df_clean.count())

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "silver_employee_scd2"
)

delta_table.alias("target").merge(
    changed_df_clean.alias("source"),
    "target.emp_id = source.emp_id AND target.is_current = true"
).whenMatchedUpdate(
    set={
        "effective_to_day": "source.snapshot_day - 1",
        "is_current": "false"
    }
).execute()

In [0]:
new_versions = (
    changed_df_clean
    .withColumn("effective_from_day", col("snapshot_day"))
    .withColumn("effective_to_day", lit(9999))
    .withColumn("is_current", lit(True))
    .drop("snapshot_day")
)

new_versions.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("silver_employee_scd2")

In [0]:
scd2_df = spark.table("silver_employee_scd2")

print("Total SCD2 Records:", scd2_df.count())

print(
    "Historical records:",
    scd2_df.filter(col("is_current") == False).count()
)

print(
    "Current records:",
    scd2_df.filter(col("is_current") == True).count()
)

In [0]:
day4_df = (
    silver_df
    .filter(
        regexp_extract("source_file", r"(day\d+)", 1) == "day4"
    )
    .withColumn("snapshot_day", lit(4))
)

print("Day 4 Records:", day4_df.count())

In [0]:
scd2_df = spark.table("silver_employee_scd2")

current_df = scd2_df.filter(
    col("is_current") == True
)

print("Current Records:", current_df.count())

In [0]:
comparison_df = (
    day4_df.alias("new")
    .join(
        current_df.alias("old"),
        col("new.emp_id") == col("old.emp_id"),
        "left"
    )
)

In [0]:
comparison_df = comparison_df.withColumn(
    "record_changed",
    col("old.emp_id").isNull() |
    (col("new.name") != col("old.name")) |
    (col("new.department") != col("old.department")) |
    (col("new.salary") != col("old.salary")) |
    (col("new.join_date") != col("old.join_date")) |
    (col("new.status") != col("old.status"))
)

print(
    "New / Changed Employees:",
    comparison_df
    .filter(col("record_changed") == True)
    .count()
)

In [0]:
changed_df_clean = comparison_df.filter(
    col("record_changed") == True
).select(
    col("new.emp_id").alias("emp_id"),
    col("new.name").alias("name"),
    col("new.department").alias("department"),
    col("new.salary").alias("salary"),
    col("new.join_date").alias("join_date"),
    col("new.status").alias("status"),
    col("new.source_file").alias("source_file"),
    col("new.snapshot_day").alias("snapshot_day")
)

print("Records to update:", changed_df_clean.count())

In [0]:
delta_table = DeltaTable.forName(
    spark,
    "silver_employee_scd2"
)

delta_table.alias("target").merge(
    changed_df_clean.alias("source"),
    "target.emp_id = source.emp_id AND target.is_current = true"
).whenMatchedUpdate(
    set={
        "effective_to_day": "source.snapshot_day - 1",
        "is_current": "false"
    }
).execute()

In [0]:
new_versions = (
    changed_df_clean
    .withColumn("effective_from_day", col("snapshot_day"))
    .withColumn("effective_to_day", lit(9999))
    .withColumn("is_current", lit(True))
    .drop("snapshot_day")
)

new_versions.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("silver_employee_scd2")

In [0]:
scd2_df = spark.table("silver_employee_scd2")

print("Total SCD2 Records:", scd2_df.count())

print(
    "Historical records:",
    scd2_df.filter(col("is_current") == False).count()
)

print(
    "Current records:",
    scd2_df.filter(col("is_current") == True).count()
)

In [0]:
day5_df = (
    silver_df
    .filter(
        regexp_extract("source_file", r"(day\d+)", 1) == "day5"
    )
    .withColumn("snapshot_day", lit(5))
)

print("Day 5 Records:", day5_df.count())

In [0]:
scd2_df = spark.table("silver_employee_scd2")

current_df = scd2_df.filter(
    col("is_current") == True
)

print("Current Records:", current_df.count())

In [0]:
comparison_df = (
    day5_df.alias("new")
    .join(
        current_df.alias("old"),
        col("new.emp_id") == col("old.emp_id"),
        "left"
    )
)

In [0]:
comparison_df = comparison_df.withColumn(
    "record_changed",
    col("old.emp_id").isNull() |
    (col("new.name") != col("old.name")) |
    (col("new.department") != col("old.department")) |
    (col("new.salary") != col("old.salary")) |
    (col("new.join_date") != col("old.join_date")) |
    (col("new.status") != col("old.status"))
)

print(
    "New / Changed Employees:",
    comparison_df
    .filter(col("record_changed") == True)
    .count()
)

In [0]:
changed_df_clean = comparison_df.filter(
    col("record_changed") == True
).select(
    col("new.emp_id").alias("emp_id"),
    col("new.name").alias("name"),
    col("new.department").alias("department"),
    col("new.salary").alias("salary"),
    col("new.join_date").alias("join_date"),
    col("new.status").alias("status"),
    col("new.source_file").alias("source_file"),
    col("new.snapshot_day").alias("snapshot_day")
)

print("Records to update:", changed_df_clean.count())

In [0]:
delta_table = DeltaTable.forName(
    spark,
    "silver_employee_scd2"
)

delta_table.alias("target").merge(
    changed_df_clean.alias("source"),
    "target.emp_id = source.emp_id AND target.is_current = true"
).whenMatchedUpdate(
    set={
        "effective_to_day": "source.snapshot_day - 1",
        "is_current": "false"
    }
).execute()

In [0]:
new_versions = (
    changed_df_clean
    .withColumn("effective_from_day", col("snapshot_day"))
    .withColumn("effective_to_day", lit(9999))
    .withColumn("is_current", lit(True))
    .drop("snapshot_day")
)

new_versions.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("silver_employee_scd2")

In [0]:
scd2_df = spark.table("silver_employee_scd2")

print("Total SCD2 Records:", scd2_df.count())

print(
    "Historical records:",
    scd2_df.filter(col("is_current") == False).count()
)

print(
    "Current records:",
    scd2_df.filter(col("is_current") == True).count()
)

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

In [0]:
display(
    spark.table("silver_employee_scd2")
)

In [0]:
display(spark.table("silver_employee_scd2"))

In [0]:
spark.table("silver_employee_scd2").printSchema()

In [0]:
display(
    spark.sql("""
        SELECT
            emp_id,
            name,
            department,
            salary,
            effective_from_day,
            effective_to_day,
            is_current
        FROM silver_employee_scd2
        ORDER BY emp_id, effective_from_day
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            emp_id,
            COUNT(*) AS total_versions,
            SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) AS current_versions
        FROM silver_employee_scd2
        GROUP BY emp_id
        ORDER BY emp_id
    """)
)